# 04 — Procesamiento con Spark sobre S3

Notebook diseñado para correr en la **EC2 con Spark + Jupyter** descrita en `infra/aws/README.md`.

Lee las predicciones que el backend envía a Firehose → S3 (`raw/predictions/`), las agrega y deja métricas listas para reentrenar o monitorear.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName('HealthAI-S3')
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4')
    .getOrCreate())
spark

In [ ]:
BUCKET = 's3a://healthai-breast-bdi-data'

# Predicciones que llegan vía Firehose (formato JSON, gzip)
preds = spark.read.json(f'{BUCKET}/raw/predictions/')
preds.printSchema()
preds.show(5, truncate=False)
print('Total predicciones:', preds.count())

In [ ]:
# Distribución de niveles de riesgo emitidos
preds.groupBy('nivel_riesgo').count().orderBy('count', ascending=False).show()

In [ ]:
# Dataset BCSC limpio cargado en S3 desde local
bcsc = spark.read.parquet(f'{BUCKET}/clean/bcsc_clean.parquet')
print('Filas:', bcsc.count())
bcsc.groupBy('breast_cancer_history').agg(F.sum('count').alias('mamografias')).show()